# KhazinaSmart — Model Training & Evaluation

Trains and evaluates Linear Regression (baseline), XGBoost (primary), and LightGBM (comparison).

In [ ]:
import sys, os
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import joblib
import plotly.graph_objects as go
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from src.feature_engineering import get_feature_columns, get_train_test_split
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/features_final.csv', parse_dates=['Date'])
X_train, X_test, y_train, y_test = get_train_test_split(df)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## Baseline: Linear Regression

In [ ]:
def metrics(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1))) * 100
    r2 = r2_score(y_true, y_pred)
    print(f'[{name}] RMSE: {rmse:,.0f} | MAE: {mae:,.0f} | MAPE: {mape:.2f}% | R2: {r2:.4f}')
    return {'Model': name, 'RMSE': round(rmse,0), 'MAE': round(mae,0), 'MAPE': round(mape,2), 'R2': round(r2,4)}

lr = LinearRegression()
lr.fit(X_train, y_train)
lr_results = metrics(y_test, lr.predict(X_test), 'Linear Regression')

## Primary Model: XGBoost

In [ ]:
xgb = XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, verbosity=0
)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_preds = xgb.predict(X_test)
xgb_results = metrics(y_test, xgb_preds, 'XGBoost')

## Comparison: LightGBM

In [ ]:
lgbm = LGBMRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, verbose=-1
)
lgbm.fit(X_train, y_train)
lgbm_results = metrics(y_test, lgbm.predict(X_test), 'LightGBM')

## Model Comparison Table

In [ ]:
results_df = pd.DataFrame([lr_results, xgb_results, lgbm_results])
print('\n=== Model Comparison ===')
print(results_df.to_string(index=False))

## Actual vs Predicted — XGBoost

In [ ]:
sample = min(300, len(y_test))
fig = go.Figure()
fig.add_trace(go.Scatter(y=y_test.values[:sample], name='Actual', line=dict(color='#4a9eff', width=2)))
fig.add_trace(go.Scatter(y=xgb_preds[:sample], name='Predicted', line=dict(color='#e94560', width=2, dash='dash')))
fig.update_layout(title='XGBoost: Actual vs Predicted Weekly Sales',
                  height=500, template='plotly_white',
                  xaxis_title='Sample Index', yaxis_title='Weekly Sales')
fig.show()

## Feature Importance — Top 20

In [ ]:
feat_names = X_train.columns.tolist()
importances = xgb.feature_importances_
fi_df = pd.DataFrame({'Feature': feat_names, 'Importance': importances})
fi_df = fi_df.sort_values('Importance', ascending=True).tail(20)
fig2 = px.bar(fi_df, x='Importance', y='Feature', orientation='h',
              title='XGBoost Feature Importance — Top 20',
              color='Importance', color_continuous_scale=['#4a9eff', '#1a1a2e'])
fig2.update_layout(height=500, template='plotly_white')
fig2.show()

## Residual Analysis

In [ ]:
residuals = y_test.values - xgb_preds
fig3 = px.histogram(x=residuals, nbins=80,
                    title='XGBoost Residuals Distribution',
                    color_discrete_sequence=['#e94560'])
fig3.add_vline(x=0, line_dash='dash', line_color='white')
fig3.update_layout(height=400, template='plotly_white',
                   xaxis_title='Residual (Actual - Predicted)', yaxis_title='Count')
fig3.show()
print(f'Residual mean: {residuals.mean():,.2f} (should be ~0)')
print(f'Residual std:  {residuals.std():,.2f}')

## Save Best Model

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(xgb, '../models/best_model.pkl')
joblib.dump(xgb, '../models/xgb_tuned.pkl')  # also save as tuned
print(f'''
=== KhazinaSmart Model Summary ===
Best Model: XGBoost
RMSE: {xgb_results["RMSE"]:,.0f} | MAE: {xgb_results["MAE"]:,.0f} | MAPE: {xgb_results["MAPE"]}% | R2: {xgb_results["R2"]}
Model saved to models/best_model.pkl
''')